In [20]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from pathlib import Path
import glob

# ============================================================
# 1. CARREGAR SILVER SINAN (todos os anos particionados)
# ============================================================
arquivos = sorted(glob.glob('../data/silver/sinan/*.parquet'))
silver = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)

print(f"Silver SINAN carregado: {silver.shape}")
print(f"Colunas disponíveis: {silver.columns.tolist()}")

Silver SINAN carregado: (390048, 9)
Colunas disponíveis: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_UNIDADE', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']


In [22]:
import requests
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import time

# ============================================================
# 2. BUSCAR CNES — paginação de 20 em 20 (método que funcionou)
# ============================================================
def buscar_todas_paginas(cod_municipio, nome):
    todos = []
    offset = 0
    while True:
        url = (f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
               f"?codigo_municipio={cod_municipio}&limit=20&offset={offset}")
        r = requests.get(url, timeout=15)
        lote = r.json().get('estabelecimentos', [])
        if not lote:
            break
        todos.extend(lote)
        offset += 20
        print(f"\r  {nome}: {offset} buscados...", end='')
        time.sleep(0.1)
    print(f"\n  ✅ Total {nome}: {len(todos)}")
    return pd.DataFrame(todos)

print("Buscando CNES — aguarde ~2 minutos...")
df_cuiaba = buscar_todas_paginas('510340', 'Cuiabá')
df_vg     = buscar_todas_paginas('510840', 'Várzea Grande')

df_cnes_full = pd.concat([df_cuiaba, df_vg], ignore_index=True)
print(f"\n✅ Total geral: {len(df_cnes_full)}")

Buscando CNES — aguarde ~2 minutos...
  Cuiabá: 2740 buscados...
  ✅ Total Cuiabá: 2727
  Várzea Grande: 380 buscados...
  ✅ Total Várzea Grande: 372

✅ Total geral: 3099


In [23]:
# ============================================================
# 3. AGREGAR CASOS POR UNIDADE (Silver SINAN)
# ============================================================
carga_unidade = (
    silver
    .groupby('ID_UNIDADE')
    .size()
    .reset_index(name='casos_historicos')
)
print(f"Unidades únicas no SINAN: {len(carga_unidade)}")

# ============================================================
# 4. FILTRAR CNES COM COORDENADAS VÁLIDAS + MERGE
# ============================================================
df_cnes_full['codigo_cnes'] = df_cnes_full['codigo_cnes'].astype(str).str.strip()

df_cnes_geo = df_cnes_full[
    df_cnes_full['latitude_estabelecimento_decimo_grau'].notna() &
    (df_cnes_full['latitude_estabelecimento_decimo_grau'] != 0)
].copy()

print(f"CNES com coordenadas válidas: {len(df_cnes_geo)}")

# Merge SINAN × CNES
df_score = carga_unidade.merge(
    df_cnes_geo[['codigo_cnes', 'nome_fantasia', 'bairro_estabelecimento',
                 'latitude_estabelecimento_decimo_grau',
                 'longitude_estabelecimento_decimo_grau',
                 'codigo_municipio', 'codigo_tipo_unidade']],
    left_on='ID_UNIDADE',
    right_on='codigo_cnes',
    how='inner'
)

print(f"Unidades cruzadas (SINAN × CNES): {len(df_score)}")

# ============================================================
# 5. SCORE V2 — Percentil rank + Baixa Confiança
# ============================================================
LIMIAR_CONFIANCA = 50

# Percentil rank elimina distorção hospital vs UBS
df_score['score_v2'] = df_score['casos_historicos'].rank(pct=True)
df_score['baixa_confianca'] = df_score['casos_historicos'] < LIMIAR_CONFIANCA

def classificar_risco(score):
    if score >= 0.80:   return 'Muito Alto'
    elif score >= 0.60: return 'Alto'
    elif score >= 0.40: return 'Moderado'
    elif score >= 0.20: return 'Baixo'
    else:               return 'Muito Baixo'

df_score['risco_v2'] = df_score['score_v2'].apply(classificar_risco)

print("\n" + "="*50)
print("SCORE V2 — Distribuição por risco (percentil rank)")
print("="*50)
print(df_score['risco_v2'].value_counts())
print(f"\nBaixa Confiança (< {LIMIAR_CONFIANCA} casos): {df_score['baixa_confianca'].sum()}")
print(f"Alta Confiança: {(~df_score['baixa_confianca']).sum()}")
print(f"\nTop 5 maior risco:")
print(df_score.nlargest(5, 'score_v2')[
    ['nome_fantasia', 'bairro_estabelecimento', 'casos_historicos', 'risco_v2', 'baixa_confianca']
].to_string())

Unidades únicas no SINAN: 1737
CNES com coordenadas válidas: 2941
Unidades cruzadas (SINAN × CNES): 191

SCORE V2 — Distribuição por risco (percentil rank)
risco_v2
Muito Alto     39
Moderado       38
Alto           38
Baixo          38
Muito Baixo    38
Name: count, dtype: int64

Baixa Confiança (< 50 casos): 107
Alta Confiança: 84

Top 5 maior risco:
                                            nome_fantasia bairro_estabelecimento  casos_historicos    risco_v2  baixa_confianca
15   HOSPITAL E PRONTO SOCORRO MUNICIPAL DE VARZEA GRANDE             CENTRO SUL             11118  Muito Alto            False
41          HOSPITAL E PRONTO SOCORRO MUNICIPAL DE CUIABA           BANDEIRANTES              7574  Muito Alto            False
28                                     POLICLINICA VERDAO                 VERDAO              6997  Muito Alto            False
27           CENTRO DE ESPECIALIDADES MEDICAS DO PLANALTO                CARUMBE              4868  Muito Alto            False
151  

In [24]:
# ============================================================
# 6. MAPA FOLIUM V2 — Score normalizado + Baixa Confiança
# ============================================================

mapa = folium.Map(
    location=[-15.62, -56.09],
    zoom_start=12,
    tiles='CartoDB positron'
)

# Cores por risco
cores = {
    'Muito Alto':  '#d73027',
    'Alto':        '#fc8d59',
    'Moderado':    '#fee090',
    'Baixo':       '#91bfdb',
    'Muito Baixo': '#4575b4'
}

# Marcadores
for _, row in df_score.iterrows():
    lat = row['latitude_estabelecimento_decimo_grau']
    lon = row['longitude_estabelecimento_decimo_grau']
    cor = cores[row['risco_v2']]
    
    # Ícone de alerta para Baixa Confiança
    confianca_txt = "⚠️ Baixa Confiança" if row['baixa_confianca'] else "✅ Alta Confiança"
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=5 + row['score_v2'] * 15,
        color='gray' if row['baixa_confianca'] else cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.4 if row['baixa_confianca'] else 0.85,
        popup=folium.Popup(
            f"<b>{row['nome_fantasia']}</b><br>"
            f"Bairro: {row['bairro_estabelecimento']}<br>"
            f"Casos históricos: {row['casos_historicos']:,}<br>"
            f"Score: {row['score_v2']:.2f}<br>"
            f"Risco: <b>{row['risco_v2']}</b><br>"
            f"{confianca_txt}",
            max_width=260
        ),
        tooltip=f"{row['nome_fantasia']} — {row['risco_v2']} {('⚠️' if row['baixa_confianca'] else '')}"
    ).add_to(mapa)

# Heatmap apenas com Alta Confiança
df_hc = df_score[~df_score['baixa_confianca']]
heat_data = df_hc[['latitude_estabelecimento_decimo_grau',
                    'longitude_estabelecimento_decimo_grau',
                    'casos_historicos']].values.tolist()
HeatMap(heat_data, radius=25, blur=15, min_opacity=0.4).add_to(mapa)

# Legenda
legenda = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background-color: white; padding: 15px; border-radius: 8px;
     border: 2px solid #ccc; font-size: 13px; line-height: 1.8;">
<b>🦟 Score de Risco — Dengue MT v2</b><br>
<i>Percentil rank por unidade de saúde</i><br><br>
🔴 Muito Alto (top 20%)<br>
🟠 Alto (60–80%)<br>
🟡 Moderado (40–60%)<br>
🔵 Baixo (20–40%)<br>
⚫ Muito Baixo (bottom 20%)<br><br>
<b>⚠️ Borda cinza = Baixa Confiança</b><br>
<small>(&lt; 50 casos históricos)</small><br><br>
<small>Fonte: SINAN/DATASUS + CNES 2007–2024</small>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda))

# Salvar
mapa.save('../reports/mapa_risco_dengue_v2.html')
print("Mapa v2 salvo em reports/mapa_risco_dengue_v2.html")

# Abrir no navegador
import webbrowser, os
caminho = os.path.abspath('../reports/mapa_risco_dengue_v2.html')
webbrowser.open(f'file:///{caminho}')

Mapa v2 salvo em reports/mapa_risco_dengue_v2.html


True

In [25]:
# ============================================================
# 7. SALVAR CNES E SCORE LOCALMENTE
# ============================================================
from pathlib import Path

Path('../data/external').mkdir(parents=True, exist_ok=True)

# Salvar CNES completo
df_cnes_full.to_parquet('../data/external/cnes_cuiaba_vg.parquet', index=False)
print(f"CNES salvo: {len(df_cnes_full)} registros")

# Salvar score v2
df_score.to_parquet('../data/external/score_risco_v2.parquet', index=False)
print(f"Score v2 salvo: {len(df_score)} unidades")

# Resumo final
print(f"\n{'='*50}")
print(f"RESUMO SCORE V2")
print(f"{'='*50}")
print(f"Total unidades mapeadas: {len(df_score)}")
print(f"Alta Confiança: {(~df_score['baixa_confianca']).sum()}")
print(f"Baixa Confiança: {df_score['baixa_confianca'].sum()}")
print(f"\nDistribuição:")
print(df_score['risco_v2'].value_counts())

CNES salvo: 3099 registros
Score v2 salvo: 191 unidades

RESUMO SCORE V2
Total unidades mapeadas: 191
Alta Confiança: 84
Baixa Confiança: 107

Distribuição:
risco_v2
Muito Alto     39
Moderado       38
Alto           38
Baixo          38
Muito Baixo    38
Name: count, dtype: int64
